In [1]:
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
from ipywidgets import interact
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt

import gplately

from lib.main_red_borders import *

from parameters import parameters

from joblib import Parallel, delayed
from matplotlib.ticker import FuncFormatter
from tqdm import tqdm

In [2]:
# Plate model name
plate_model_name = parameters["plate_model_name"]

# Timespan for analysis
temporal_resolution = 1
time_min = 0
time_max = 5
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

plate_model_dir = parameters["plate_model_dir"]
outputs_dir = parameters["outputs_dir"]
feat_maps_dir = parameters["feat_maps_dir"]

if not os.path.exists(outputs_dir):
    os.makedirs(outputs_dir, exist_ok=True)

feat_maps_dir = os.path.join(outputs_dir, feat_maps_dir)

subduction_data_filename = parameters["subduction_data_filename"]
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)

nprocs = 12

In [3]:

coastlines_filename = "StaticGeometries/Coastlines/Global_coastlines_low_res.shp"
coastlines = os.path.join(plate_model_dir, coastlines_filename)
continents_filename = "StaticGeometries/ContinentalPolygons/Global_EarthByte_GPlates_PresentDay_ContinentsAndArcs.shp"
continents = os.path.join(plate_model_dir, continents_filename)
COBs_filename = "StaticGeometries/COBLineSegments/Global_EarthByte_GeeK07_COBLineSegments_2019_v1.shp"
COBs = os.path.join(plate_model_dir, COBs_filename)

plate_model = get_plate_reconstruction(
    model_name=plate_model_name,
    model_dir=plate_model_dir,
)

if os.path.isfile(subduction_data_filename):
    subduction_data = pd.read_csv(subduction_data_filename)
else:
    subduction_data = run_calculate_convergence(
        nprocs=nprocs,
        min_time=time_min,
        max_time=time_max,
        temporal_resolution=temporal_resolution,
        plate_reconstruction=plate_model,
        verbose=True,
    )
    
    subduction_data.to_csv(subduction_data_filename, index=False)

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   2 out of   6 | elapsed:    9.0s remaining:   18.0s
[Parallel(n_jobs=12)]: Done   3 out of   6 | elapsed:    9.0s remaining:    9.0s
[Parallel(n_jobs=12)]: Done   4 out of   6 | elapsed:    9.1s remaining:    4.5s
[Parallel(n_jobs=12)]: Done   6 out of   6 | elapsed:    9.1s finished


In [4]:
subduction_data_columns = subduction_data.columns.tolist()
features_plot = subduction_data_columns.copy()
features_plot.remove("lon")
features_plot.remove("lat")
features_plot.remove("age (Ma)")
features_plot.remove("subducting_plate_ID")
features_plot.remove("trench_plate_ID")

gplot = gplately.PlotTopologies(plate_model, coastlines, continents, COBs)

projection = ccrs.Mollweide(central_longitude=60)

In [18]:
@interact
def show_map(time=time_steps, feature=features_plot):
    gplot.time = time
    
    subduction_data_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = plt.axes(projection=projection, facecolor="azure")

    gplot.plot_continents(ax, edgecolor="none", facecolor="tan", alpha=0.5, zorder=1)
    gplot.plot_coastlines(ax, edgecolor="none", facecolor="tan", alpha=0.7, zorder=2)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)
    
    feat = ax.scatter(subduction_data_t["lon"], subduction_data_t["lat"], 50, marker=".",
                      c=subduction_data_t[feature], cmap=ccm.hawaii_r, transform=ccrs.PlateCarree(), zorder=4)

    gplot.plot_all_topologies(ax, color="orangered", zorder=5)
    gplot.plot_trenches(ax, color="black", alpha=0.7, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color="black", alpha=0.3, zorder=7)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)
    
    ax.text(0.49,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, "60°W", transform=ax.transAxes, fontsize=16)
    
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}
        
    cbar_feat = fig.colorbar(feat, shrink=0.4, pad=0.06, orientation="horizontal", extend="both")
    cbar_feat.set_label(feature, fontsize=16, labelpad=10)
    cbar_feat.ax.tick_params(labelsize=16)
    
    # Define custom legend handles
    custom_handles = [
        Patch(facecolor="tan", edgecolor="none", label="Continental Crust"),
        #Line2D([0], [0], color="dimgray", lw=2, label="Mid-Ocean Ridge")
        Line2D([0], [0], color="orangered", label="MORs/Transforms"),
        Line2D([0], [0], color="black", label="Trench Lines"),
    ]

    # Add the custom legend to the plot
    ax.legend(handles=custom_handles, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.2))
    
    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
        
    plt.show()

interactive(children=(Dropdown(description='time', options=(0, 1, 2, 3, 4, 5), value=0), Dropdown(description=…

In [5]:
if os.path.exists(feat_maps_dir):
    print(f"Feature maps are located in {feat_maps_dir}")
else:
    os.makedirs(feat_maps_dir, exist_ok=True)
    
    generate_feat_maps(
        gplot,
        subduction_data,
        projection,
        time_steps,
        feature="convergence_rate (cm/yr)",
        output_dir=feat_maps_dir,
        num_jobs=nprocs
    )
    
    output_filenames = [
        os.path.join(feat_maps_dir, f"feat_map_{t:0.0f}Ma.png")
        for t in time_steps
    ]

Dispatching tasks: 100%|██████████| 6/6 [00:00<00:00, 3041.19it/s]
